In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

In [8]:
data = pd.read_csv('/Users/sid/Downloads/creditcard.csv')
data.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')

In [9]:
X = data.drop('Class', axis=1)
y = data['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression())
])

In [10]:
pipe.fit(X_train, y_train)
pipe.predict(X_test)

array([0, 0, 0, ..., 0, 0, 0])

In [13]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='average_precision')
print(scores.mean(), scores.std())

0.7614969310778783 0.04111280238857698


In [15]:
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Test Average Precision:", average_precision_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.63      0.72        98

    accuracy                           1.00     56962
   macro avg       0.91      0.82      0.86     56962
weighted avg       1.00      1.00      1.00     56962

[[56851    13]
 [   36    62]]
Test Average Precision: 0.7413820992780461


In [16]:
# in comparision leaky model
# WRONG: fit scaler on full X before splitting (leakage)
leaky_scaler = StandardScaler().fit(X)  # sees test data statistics
X_leaky = leaky_scaler.transform(X)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42, stratify=y)

leaky_model = LogisticRegression().fit(X_train_l, y_train_l)
print("Leaky:", average_precision_score(y_test_l, leaky_model.predict_proba(X_test_l)[:,1]))
print("Correct (pipeline):", average_precision_score(y_test, y_proba))

Leaky: 0.742400961705607
Correct (pipeline): 0.7413820992780461
